In [ ]:
import deepfmkit.core as dfm
from deepfmkit.plotting import default_rc
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

# Instantiate the main framework
dff = dfm.DeepFrame()

# --- 1. Define the Single, Shared Laser Source ---
laser_config = dfm.LaserConfig(label="main_laser")
laser_config.fm = 1000  # Modulation frequency (Hz)
laser_config.f_n = 1e6  # Laser frequency noise at 1 Hz (Hz/rtHz)
laser_config.r_n = 1e-5  # Laser relative intensity noise (1/rtHz)

# --- 2. Define the Main Interferometer ---
main_ifo_config = dfm.IfoConfig(label="dynamic_ifo")
main_ifo_config.ref_arml = 0.1  # Reference arm length (m)
main_ifo_config.meas_arml = 0.3  # Measurement arm length (m)
main_ifo_config.arml_mod_f = 1.0  # Measurement arm modulation frequency (Hz)
main_ifo_config.arml_mod_amp = 0.0  # Armlength modulation amplitude (m)
main_ifo_config.arml_n = 0.0  # Armlength noise (m/rtHz)

# --- 3. Set Modulation Depth by Adjusting Laser's `df` ---
m_target = 6.0  # Target effective modulation index (rad)
laser_config.set_df_for_effect(main_ifo_config, m_target)

# --- 4. Compose the Main Channel ---
main_label = "dynamic_channel"
main_channel = dfm.SimConfig(
    label=main_label,
    laser_config=laser_config,
    ifo_config=main_ifo_config,
    f_samp=int(200e3),  # Sampling frequency (Hz)
)
dff.sims[main_label] = main_channel

# --- 6. Simulate ---
dff.simulate(
    label=main_label,
    n_seconds=5,  # Simulation length in seconds
    verbose=True,
)

# NLS fit
dff.fit(main_label, fit_label="nls", n=20, method="nls")

In [ ]:
# Run the EKF with default tuning parameters
dff.fit(main_label, fit_label="ekf", method="ekf")

# Run with custom tuning
custom_Q = [1e-9, 1e-9, 1e-7, 1e-7, 1e-9]
custom_R = 0.001
dff.fit(
    main_label, fit_label="ekf_tuned", method="ekf", Q_diag=custom_Q, R_val=custom_R
)

In [ ]:
ax = dff.plot(
    labels=["nls", "ekf", "ekf_tuned"],
    which=["m", "phi"])
plt.show()